# MehuLLM — voice-layer LoRA on a free Colab T4

Trains **Qwen3-1.7B** to rewrite a neutral English draft into Mehul's WhatsApp voice.

**Runtime → Change runtime type → T4 GPU** before running anything.

### Why fp16 LoRA and not QLoRA

A 1.7B model in fp16 is ~3.4 GB on a 16 GB T4. 4-bit buys nothing here except
quantisation error and `bitsandbytes` version pain. QLoRA is for when the model
does not fit — this one fits comfortably.

### Why these exact flags

A T4 is **sm_75** (Turing), the same architecture class as the GTX 1650 this will
eventually run on. That means:

* `fp16=True, bf16=False` — Turing has no bf16 tensor cores. Setting `bf16=True`
  fails or silently falls back.
* `attn_implementation="sdpa"` — FlashAttention-2 requires Ampere (sm_80+).
* `optim="adamw_8bit"` — halves optimiser state memory at no quality cost.

### Colab **will** disconnect

Free sessions cap out around 3–4 h and this run is ~2.5–3 h wall clock. Checkpoints
go straight to Drive every 200 steps and the trainer resumes automatically. If you
get disconnected, just re-run every cell — it picks up where it stopped.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
major, minor = torch.cuda.get_device_capability()
print(f"compute capability {major}.{minor}")
if major < 8:
    print("sm_75 (Turing) -> fp16 only, no bf16, SDPA attention. Expected on a T4.")
else:
    print(f"sm_{major}{minor} -> bf16 available, but we keep fp16 so training\n"
          "matches the sm_75 card this model will actually be served on.")

## 2 · Install

Unsloth is used mainly because it **pins a self-consistent stack**. The speed and
memory wins are a bonus; the real value is not fighting `transformers` v5 /
`trl` API churn (`tokenizer=` → `processing_class=`, `max_seq_length` →
`max_length`, peft ≥ 0.18 requirements) on a machine you re-provision every session.

**After the first green run**, freeze what actually worked and install from that
file every time afterwards — the last cell writes it for you.

In [ ]:
%%capture
import os

LOCK = "/content/drive/MyDrive/mehullm/colab_train.lock.txt"

if os.path.exists(LOCK):
    print("installing from lockfile")
    !pip install -q -r {LOCK}
else:
    !pip install -q unsloth
    !pip install -q --no-deps --upgrade "trl>=0.15" peft accelerate bitsandbytes

## 3 · Mount Drive and locate the dataset

Upload `data/derived/sft_pairs.jsonl` from your laptop to
`MyDrive/mehullm/sft_pairs.jsonl` first.

That file contains **your scrubbed private messages**. It is PII-scrubbed and
name-pseudonymised, but it is still yours — keep the Drive folder private and do
not share the notebook with outputs saved.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

WORK = "/content/drive/MyDrive/mehullm"
DATA = f"{WORK}/sft_pairs.jsonl"
CKPT = f"{WORK}/ckpt"
os.makedirs(CKPT, exist_ok=True)

assert os.path.exists(DATA), (
    f"{DATA} not found.\n"
    "Upload data/derived/sft_pairs.jsonl from your laptop to MyDrive/mehullm/"
)
print(f"dataset: {os.path.getsize(DATA) / 1e6:.1f} MB")

## 4 · Load and sanity-check the data

**Do not skip this.** The single most damaging failure mode for this project is a
*degenerate* dataset — one where the draft is a copy of the reply. Training loss
would look perfectly healthy while the model learns the identity function, and
you would only find out at evaluation. The pipeline filters these, so a non-zero
count here means something regressed upstream.

In [ ]:
import json
import re
from collections import Counter

rows = [json.loads(l) for l in open(DATA, encoding="utf-8") if l.strip()]
print(f"{len(rows):,} rows")
print("split   :", Counter(r["split"] for r in rows))
print("variant :", Counter(r["variant"] for r in rows))
print("bucket  :", Counter(r["bucket"] for r in rows))


def norm(s):
    return re.sub(r"[^\w]", "", s).casefold()


degenerate = 0
for r in rows:
    draft = re.search(r"<draft>\n(.*?)\n</draft>", r["messages"][1]["content"], re.S)
    if draft and norm(draft.group(1)) == norm(r["messages"][2]["content"]):
        degenerate += 1

print(f"\ndegenerate (draft == reply): {degenerate}")
assert degenerate == 0, (
    "Degenerate pairs teach the identity function. Loss will look fine and the "
    "model will learn nothing. Fix the neutraliser before training."
)

print("\n--- one training example ---")
for m in rows[0]["messages"]:
    print(f"[{m['role']}] {m['content'][:220]}")

In [ ]:
from datasets import Dataset

# Split by the label the pipeline assigned. It split BY CHAT, not by message,
# so no conversation appears on both sides -- otherwise val loss is meaningless.
train_rows = [{"messages": r["messages"]} for r in rows if r["split"] == "train"]
val_rows = [{"messages": r["messages"]} for r in rows if r["split"] == "val"]

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)
print(f"train {len(train_ds):,}   val {len(val_ds):,}")

## 5 · Load Qwen3-1.7B

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ = 1024  # context + draft + reply fits well inside this

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=MAX_SEQ,
    dtype=torch.float16,  # sm_75: fp16 only
    load_in_4bit=False,   # 1.7B fits in fp16 on a 16 GB T4
)
print(f"loaded, {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    # All attention AND MLP projections. Style lives in the MLPs as much as in
    # attention -- attention-only adapters underfit register/tone transfer.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## 6 · Turn thinking OFF

Qwen3 is a hybrid-thinking model. Left alone it emits `<think>…</think>` blocks,
which would be baked into the adapter and then leak into WhatsApp replies at
inference. Verify the rendered template contains an **empty** think block.

In [ ]:
sample = tokenizer.apply_chat_template(
    train_ds[0]["messages"], tokenize=False, enable_thinking=False
)
print(sample[:900])

import re

blocks = re.findall(r"<think>(.*?)</think>", sample, re.S)
assert all(not b.strip() for b in blocks), (
    "Non-empty <think> block in the rendered template -- thinking would be "
    "trained in and leak into replies."
)
print("\nthinking suppressed OK")

## 7 · Train

Loss is computed **only on Mehul's reply** — not on the system prompt, the
context, or the neutral draft. Without that masking most of the gradient goes
into learning to reproduce the draft, which is exactly backwards.

We do the masking explicitly (`labels = -100` on the prompt) rather than via
TRL's `assistant_only_loss`, because Unsloth's patched `SFTTrainer` will not
auto-detect a `messages` column and the marker-based path fights it. Doing it by
hand also means the mask is printed and asserted before training starts.

The reply's `<|im_end|>` **is** supervised, so the model learns where to stop.

`save_steps=200` writes to Drive, not local disk — local disk dies with the
session.


In [ ]:
# --- EXPLICIT TOKENISATION + LABEL MASK ------------------------------------
# We build input_ids and labels ourselves instead of relying on
# `assistant_only_loss` + {% generation %} markers. Two reasons:
#   1. Unsloth's patched SFTTrainer does not auto-detect the `messages` column
#      and demands a formatting_func, which is incompatible with the marker path.
#   2. Doing it by hand means the loss mask is verifiable rather than inferred --
#      and the reply's <|im_end|> IS included, so the model learns to STOP.
#      (Leaving it out is why TRL warned "the model may not learn to stop".)
#
# The prompt is byte-identical to what the Ollama Modelfile prefills at
# inference: an empty <think> block before the reply.
IM_END = "<|im_end|>"

def _prompt_text(msgs):
    sys_, usr = msgs[0]["content"], msgs[1]["content"]
    return (
        f"<|im_start|>system\n{sys_}{IM_END}\n"
        f"<|im_start|>user\n{usr}{IM_END}\n"
        f"<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )

def encode(row):
    msgs = row["messages"]
    p_ids = tokenizer(_prompt_text(msgs), add_special_tokens=False)["input_ids"]
    r_ids = tokenizer(msgs[2]["content"] + IM_END, add_special_tokens=False)["input_ids"]
    ids = (p_ids + r_ids)[:MAX_SEQ]
    # -100 tells the loss to ignore those positions: no loss on system, context
    # or draft -- only on the reply we want the model to imitate.
    labels = ([-100] * len(p_ids) + r_ids)[:MAX_SEQ]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

train_tok = train_ds.map(encode, remove_columns=train_ds.column_names)
val_tok = val_ds.map(encode, remove_columns=val_ds.column_names)

# Verify the mask before spending three hours on it.
_ex = train_tok[0]
_sup = sum(1 for x in _ex["labels"] if x != -100)
print(f"sequence {len(_ex['input_ids'])} tokens | supervised {_sup} ({_sup/len(_ex['input_ids']):.0%})")
assert 0 < _sup < len(_ex["input_ids"]), "label mask is degenerate"
assert _ex["labels"][-1] != -100, "reply end not supervised -- model would not learn to stop"
print("SUPERVISED TEXT:", repr(tokenizer.decode([x for x in _ex["labels"] if x != -100])[:180]))
print("(must be the reply + <|im_end|>, nothing else)")


In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=CKPT,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch 16
    num_train_epochs=3,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=30,
    max_grad_norm=0.3,               # fp16 on sm_75 -- keeps loss off NaN
    weight_decay=0.01,
    fp16=True,
    bf16=False,                      # Turing has no bf16
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=200,                  # -> Drive; Colab WILL disconnect
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    seed=3407,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    # pads input_ids with pad_token and labels with -100, which is exactly what
    # a masked-label dataset needs.
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
steps = len(trainer.get_train_dataloader()) * args.num_train_epochs
print(f"{steps:.0f} optimiser steps over {args.num_train_epochs} epochs")


In [ ]:
import glob

# Resume automatically if a previous session was cut off.
resume = bool(glob.glob(f"{CKPT}/checkpoint-*"))
print("resuming from checkpoint" if resume else "starting fresh")

stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

### If loss goes NaN

fp16 on sm_75 can overflow. In order of what to try:

1. `learning_rate=5e-5` (from 1e-4)
2. `max_grad_norm=0.1` (from 0.3)
3. `per_device_train_batch_size=2, gradient_accumulation_steps=8`

Do **not** switch to bf16 — the T4 does not support it, and matching the serving
card's precision is deliberate.

## 8 · Eyeball it before exporting

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM = rows[0]["messages"][0]["content"]
PROBES = [
    ("Person_A: kal aa raha hai?", "I will let you know by tonight, definitely."),
    ("Person_A: paise chahiye the", "Please transfer 500 rupees to <UPI>."),
    ("Person_A: aaj shaam free ho?",
     "Unfortunately I am not available this evening. Would tomorrow at 6 work?"),
]

for ctx, draft in PROBES:
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",
         "content": f"<context>\n{ctx}\n</context>\n<draft>\n{draft}\n</draft>"},
    ]
    ids = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, enable_thinking=False, return_tensors="pt"
    ).to("cuda")
    out = model.generate(ids, max_new_tokens=80, temperature=0.85, top_p=0.9,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
    reply = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    print(f"draft : {draft}")
    print(f"voiced: {reply.strip()}\n")

**What good looks like:** shorter than the draft, lowercase, Hinglish, `<UPI>`
still intact. **What bad looks like:** the draft echoed back unchanged (adapter
did not take), or the placeholder mangled (facts not preserved — the runtime
invariant firewall will catch it, but it means quality is poor).

## 9 · Save the adapter and freeze the environment

In [ ]:
ADAPTER = f"{WORK}/adapter"
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print(f"adapter -> {ADAPTER}")

# Freeze the stack that actually worked. Install from this next time rather
# than re-resolving -- transformers/trl/peft churn breaks notebooks silently.
!pip freeze > {WORK}/colab_train.lock.txt
print(f"lockfile -> {WORK}/colab_train.lock.txt  (commit this to the repo)")

---

Next: **`03_merge_gguf.ipynb`** — merge the adapter, convert to GGUF Q4_K_M, and
produce the Ollama `Modelfile`. Deliberately a separate notebook against a
pinned llama.cpp commit, so a broken conversion can never cost you a training run.